# Notebook 2 — Pretraining Dataset Preparation
**Adapted for Jupyter Notebook / Kubeflow**

Sources text from HuggingFace Hub and GitHub, then applies four production-grade
data cleaning steps. Saves the cleaned corpus as `./data/preprocessed_dataset.parquet`.

**Learning objectives:**
1. Source pretraining data from HuggingFace and GitHub
2. Contrast pretraining data (raw text) vs fine-tuning data (instruction–response pairs)
3. Apply four cleaning steps: length filter, repetition filter, deduplication, language filter
4. Save cleaned data in Parquet format for downstream tokenisation

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "datasets", "langdetect", "-q"])

## 1. Sourcing Datasets

### 1a. WikiText-103 — Download from HuggingFace Hub

In [ ]:
import datasets, os

# Load WikiText-103 — clean, encyclopaedic English from Wikipedia
pretraining_dataset = datasets.load_dataset(
    "Salesforce/wikitext",
    name="wikitext-103-raw-v1",
    split="train"
)
print(pretraining_dataset)
print("\nColumns:", pretraining_dataset.column_names)

In [ ]:
# Preview a sample
sample = next(x for x in pretraining_dataset if len(x["text"]) > 100)
print("Sample text (first 500 chars):")
print(sample["text"][:500])

In [ ]:
# Keep only the text column and take 10,000 samples for demo speed
pretraining_dataset = pretraining_dataset.select_columns(["text"])
pretraining_dataset = pretraining_dataset.select(range(10_000))
print(f"Working with {pretraining_dataset.num_rows:,} samples")

### 1b. Compare: Pretraining vs Fine-tuning Data

In [ ]:
# Load Alpaca-GPT4 — a structured instruction-response dataset used for SFT
instruction_dataset = datasets.load_dataset("c-s-ale/alpaca-gpt4-data", split="train")
print(instruction_dataset)

In [ ]:
i = 0
print("── Pretraining sample (raw text, no labels) ──────────────────────")
print(pretraining_dataset[10]["text"][:300])

print("\n── Fine-tuning sample (structured instruction → response) ────────")
print("Instruction:", instruction_dataset[i]["instruction"])
print("Input      :", instruction_dataset[i]["input"])
print("Output     :", instruction_dataset[i]["output"][:200])

**Key distinction:**
- **Pretraining data** = raw unstructured text — no labels needed
- **Fine-tuning data** = structured instruction → response pairs

Pretraining gives broad world knowledge; fine-tuning aligns behaviour.
You cannot fine-tune a model that was never pretrained.

### 1c. Fetch Python Code from GitHub

In [ ]:
import requests

code_dir = "./code"
os.makedirs(code_dir, exist_ok=True)

urls = [
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/searches/double_linear_search_recursion.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/sorts/bubble_sort.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/maths/fibonacci.py",
    "https://raw.githubusercontent.com/TheAlgorithms/Python/master/data_structures/linked_list/singly_linked_list.py",
    "https://raw.githubusercontent.com/PaliC/pytorch/master/test/fx/test_subgraph_rewriter.py",
]

for url in urls:
    fname = url.split("/")[-1]
    print(f"Fetching: {fname}")
    r = requests.get(url, timeout=10)
    with open(os.path.join(code_dir, fname), "wb") as f:
        f.write(r.content)

print("\nFiles saved:", os.listdir(code_dir))

In [ ]:
code_dataset = []
for fname in os.listdir(code_dir):
    with open(os.path.join(code_dir, fname), "r", errors="ignore") as f:
        code_dataset.append({"text": f.read()})

code_dataset = datasets.Dataset.from_list(code_dataset)
print(code_dataset)

### 1d. Combine Text and Code

In [ ]:
dataset = datasets.concatenate_datasets([pretraining_dataset, code_dataset])
print(f"Combined dataset: {dataset.num_rows:,} rows")
print(f"Rows before cleaning: {dataset.num_rows:,}")

## 2. Data Cleaning

| Step | What it removes |
|---|---|
| **Length filter** | Documents shorter than 50 characters |
| **Repetition filter** | Documents with >30% duplicate paragraphs or >20% duplicate characters |
| **Deduplication** | Exact-duplicate documents across the corpus |
| **Language filter** | Non-English documents (ASCII ratio < 90%) |

### 2.1 Filter Short Documents

In [ ]:
def paragraph_length_filter(x):
    """Remove documents shorter than 50 characters."""
    return len(x["text"].strip()) >= 50

dataset = dataset.filter(paragraph_length_filter, load_from_cache_file=False)
print(f"After length filter: {dataset.num_rows:,} rows")

### 2.2 Remove Intra-document Repetitions

In [ ]:
import re

def find_duplicates(paragraphs):
    unique_p, dup_chars, dup_count = set(), 0, 0
    for p in paragraphs:
        if p in unique_p:
            dup_chars += len(p)
            dup_count += 1
        else:
            unique_p.add(p)
    return dup_count, dup_chars

def paragraph_repetition_filter(x):
    text       = x["text"]
    paragraphs = re.compile(r"\n{2,}").split(text.strip())
    if len(paragraphs) == 0:
        return False
    dup_paragraphs, dup_chars = find_duplicates(paragraphs)
    if dup_paragraphs / len(paragraphs) > 0.3:
        return False
    if dup_chars / len(text) > 0.2:
        return False
    return True

dataset = dataset.filter(paragraph_repetition_filter, load_from_cache_file=False)
print(f"After repetition filter: {dataset.num_rows:,} rows")

### 2.3 Deduplication — Exact Match

In [ ]:
def deduplication(ds):
    """Remove exact duplicate documents."""
    seen = set()
    def is_unique(x):
        if x["text"] in seen:
            return False
        seen.add(x["text"])
        return True
    return ds.filter(is_unique, load_from_cache_file=False, num_proc=1)

dataset = deduplication(dataset)
print(f"After deduplication: {dataset.num_rows:,} rows")

### 2.4 Language Filter — Keep English

In [ ]:
def english_language_filter(x):
    """Keep documents where >90% of characters are ASCII (reliable English proxy)."""
    text = x["text"].strip()
    if len(text) < 100:
        return True  # too short to judge — keep
    ascii_count = sum(1 for c in text if ord(c) < 128)
    return (ascii_count / len(text)) > 0.9

dataset = dataset.filter(english_language_filter, load_from_cache_file=False, num_proc=1)
print(f"After language filter: {dataset.num_rows:,} rows")
print(f"\n✅ Final clean dataset: {dataset.num_rows:,} rows")
print(dataset)

## 3. Save the Dataset to Disk

In [ ]:
# ── Jupyter / Kubeflow: save locally ─────────────────────────────────
# All files are saved to ./data/ in your notebook's working directory
# On Kubeflow your workspace PVC persists this folder between sessions

data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

file_path = os.path.join(data_dir, "preprocessed_dataset.parquet")
dataset.to_parquet(file_path)
print(f"✅ Saved to: {file_path}")

In [ ]:
# Reload and verify
dataset_check = datasets.Dataset.from_parquet(file_path)
print(f"✅ Verified: {dataset_check.num_rows:,} rows reloaded from {file_path}")
print(dataset_check)

## Summary

| Concept | Detail |
|---|---|
| Pretraining data | Raw unstructured text — no labels needed |
| Fine-tuning data | Structured instruction–response pairs |
| Length filter | Removes trivially short / empty documents |
| Repetition filter | Removes copy-paste spam within a document |
| Deduplication | Removes identical documents across the corpus |
| Language filter | Retains only English (ASCII > 90%) |
| Parquet format | Efficient columnar binary format — faster than CSV |
| Output | `./data/preprocessed_dataset.parquet` → input to Notebook 3 |